# Baseline — AI Hallucination Detector

**Competition:** given a `question`, a `support` passage, and a `proposed_answer`
(possibly written by a hallucinating AI), predict whether the answer is **supported by
the passage** (`label` 1) or a **hallucination** (`label` 0).

- **Task:** binary classification (10,000 balanced training rows)
- **Metric:** accuracy
- **Kaggle link:** _TODO: add link_

**Approach:** TF-IDF over the three text fields plus simple **overlap features**
(how much of the answer actually appears in the support passage) with Logistic
Regression.

In [1]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import accuracy_score

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape)
train.head(2)

(10000, 5) (3000, 4)


,row_id,question,support,proposed_answer,label
0,train_04642_0,What medical emergency occurs when a blood clo...,A stroke occurs when a blood clot blocks blood...,stroke,1
1,train_02666_1,What does the driving of turbines by the heati...,Nuclear reactors heat water to steam to drive ...,diffusion of electricity,0


In [2]:
def overlap_features(df):
    feats = []
    for _, r in df.iterrows():
        ans = set(str(r["proposed_answer"]).lower().split())
        sup = set(str(r["support"]).lower().split())
        que = set(str(r["question"]).lower().split())
        inter = len(ans & sup)
        feats.append([
            inter / max(len(ans), 1),                    # answer tokens found in support
            inter / max(len(sup), 1),
            len(ans), len(sup),
            len(ans & que) / max(len(ans), 1),
        ])
    return np.array(feats)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000)
vec.fit(pd.concat([train["question"], train["support"], train["proposed_answer"],
                   test["question"], test["support"], test["proposed_answer"]]).astype(str))

def featurize(df):
    X = sparse.hstack([
        vec.transform(df["question"].astype(str)),
        vec.transform(df["support"].astype(str)),
        vec.transform(df["proposed_answer"].astype(str)),
        sparse.csr_matrix(overlap_features(df)),
    ]).tocsr()
    return X

X, Xt, y = featurize(train), featurize(test), train["label"].values
print(X.shape)

(10000, 234683)


In [3]:
clf = LogisticRegression(max_iter=2000, C=1.0)
oof = cross_val_predict(clf, X, y, cv=5)
print(f"CV accuracy: {accuracy_score(y, oof):.4f}")

CV accuracy: 0.8122


In [4]:
clf.fit(X, y)
sub = pd.DataFrame({"row_id": test["row_id"], "label": clf.predict(Xt)})
sub.to_csv("submission.csv", index=False)
sub.head()

,row_id,label
0,test_01094_0,1
1,test_00806_0,1
2,test_00648_0,0
3,test_00808_0,1
4,test_00570_0,1


## Ideas to improve

- **NLI cross-encoders** (e.g. DeBERTa fine-tuned on MNLI): score
  `support ⇒ answer` as entailment — this is exactly the hallucination question.
- Sentence-embedding cosine between answer and each support sentence (max/mean).
- Numeric/date consistency checks: hallucinated answers often get numbers wrong.
